# Train the autoregressive box model on Colab (BIG preset)

One-click GPU run for the **A3** plan (autoregressive CAD-as-language model).
This version uses a bigger model + more data + more epochs than the first cut,
after the small preset hit F1=0.899 but only **64% exact-match** (the rest were
catastrophic, not 'one wrong edge'). Hypothesis: capacity + training time.

**Setup:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

What this does (~15–25 min on a T4):
1. Clone the repo, install deps, verify GPU.
2. Generate 10,000 training + 400 held-out boxes (~4 min).
3. Train for 100 epochs with d_model=256, 5 decoder layers (~10–18 min on T4).
4. Score under F1 + exact-match-rate; compare against the small-preset numbers.
5. Beam-3 ablation on the same checkpoint.
6. Render before/after wireframes.
7. (Optional) Save the checkpoint to Google Drive.

**Targets the run will check:**

| Predictor on held-out boxes | F1 | Exact-match |
|-----|-----|-----|
| classical baseline | 0.000 | 0.000 |
| set predictor + structure post-process | 0.282 | n/a |
| AR **small preset** (previous run) | 0.899 | **0.642** |
| AR **big preset** (this run, target) | — | **>= 0.85** |


## 1. Clone the repo

Edit `REPO_URL` below to point at your fork/push of the project.


In [ ]:
REPO_URL = 'https://github.com/Nollis/lines.git'
BRANCH = 'main'
REPO_DIR = '/content/lines'

import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print('repo at', os.getcwd())
print(subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip())


## 2. Install deps + verify GPU


In [ ]:
!pip install -q -e . matplotlib
import torch
print('torch', torch.__version__, '|  CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('NOTE: no GPU. Runtime > Change runtime type > T4 GPU, then Run all again.')


## 3. Regenerate the data (deterministic seeds -> identical content every time)


In [ ]:
# BIG preset: 10k train (vs 4k in the small preset). Disjoint dir so a previous
# small-preset data dir on disk doesn't shadow this. ~4 min on Colab CPU.
from pathlib import Path
import subprocess
TRAIN_DIR = 'data/train64_box_big'
TEST_DIR  = 'data/test64_box'
for split, n, seed, extra in [
    (TRAIN_DIR, 10000, 0, []),
    (TEST_DIR,    400, 900_000, ['--no-randomize']),
]:
    if (Path(split) / 'manifest.json').exists():
        print(f'{split}: already on disk, skipping')
        continue
    print(f'generating {split} ({n} samples)...')
    subprocess.check_call(['python', 'scripts/build_box_data.py',
                            '--out', split, '--n', str(n),
                            '--seed0', str(seed), '--canvas-side', '64'] + extra)


## 4. Train (BIG preset: 100 epochs, ~10–18 min on T4)

Calls the training function directly so the loss log streams into this cell.
If the cell dies mid-run (Colab disconnect, etc.) the per-5-epoch checkpoint
in `OUT_DIR` is reloadable with `--init-from`.


In [ ]:
from pathlib import Path
from lines.train.train_autoregressive import ARTrainConfig, train_autoregressive

OUT_DIR = Path('checkpoints/ar_box64_gpu_big')

cfg = ARTrainConfig(
    canvas_side=64,
    epochs=100,             # was 50 in the small preset
    batch_size=64,
    d_model=256,            # was 192 in the small preset
    n_decoder_layers=5,     # was 3 in the small preset
    lr=3e-4,
    device='cuda',
    checkpoint_every=5,
)
result = train_autoregressive(
    cfg,
    train_dir=Path(TRAIN_DIR),
    test_dir=Path(TEST_DIR),
    out_dir=OUT_DIR,
)


## 5. Score under the honest F1 metric + per-image distribution

Mean F1 averages over edges and hides the **distribution** of per-image quality.
A 60% perfect / 40% catastrophic mix can still average F1=0.9, but only ~60% of
drawings would be *usable*. So we report two extras:

* `exact_match_rate` = fraction of held-out drawings reconstructed completely (F1 >= 0.99)
* `near_match_rate`  = fraction with at most ~1 wrong edge per 10 (F1 >= 0.9)


In [ ]:
import torch
from lines.datagen.dataset import Dataset
from lines.datagen.sampler2d import Canvas
from lines.eval.harness import run_predictor
from lines.models.autoregressive import AutoregressiveModel
from lines.train.predictor_ar import AutoregressivePredictor
from lines.baselines.classical import ClassicalBaseline

ck = torch.load(OUT_DIR / 'model.pt', map_location='cuda', weights_only=False)
cfg = ck['cfg']
model = AutoregressiveModel(canvas_side=cfg['canvas_side'], d_model=cfg['d_model'],
                            n_heads=cfg['n_heads'], n_decoder_layers=cfg['n_decoder_layers'],
                            max_seq_len=cfg['max_seq_len']).cuda()
model.load_state_dict(ck['model'])

C = Canvas(cfg['canvas_side'], cfg['canvas_side'])
ds = Dataset(TEST_DIR)
ar_greedy = AutoregressivePredictor(model, C, max_tokens=cfg['max_seq_len'],
                                     device='cuda', beam_size=1)
r_greedy = run_predictor(ar_greedy, ds, C)
r_base = run_predictor(ClassicalBaseline(), ds, C)

print('Predictor                       F1       Exact   Near')
print(f'classical baseline              {r_base["mean_f1"]:.3f}    {r_base["exact_match_rate"]:.3f}   {r_base["near_match_rate"]:.3f}')
print(f'set predictor + post-process    0.282    (recorded)')
print(f'AR small preset (recorded)      0.899    0.642   0.650')
print(f'AR BIG greedy (this run)        {r_greedy["mean_f1"]:.3f}    {r_greedy["exact_match_rate"]:.3f}   {r_greedy["near_match_rate"]:.3f}')
print()
print(f'  perfect drawings: {r_greedy["n_perfect"]}/{r_greedy["n"]}'
      f'  ({r_greedy["exact_match_rate"]*100:.1f}%)')
print(f'  near-perfect:     {r_greedy["n_near"]}/{r_greedy["n"]}'
      f'  ({r_greedy["near_match_rate"]*100:.1f}%)')
ar = ar_greedy   # used by the visualization cell below
r_ar = r_greedy

exact_target = 0.85
delta_exact = r_greedy['exact_match_rate'] - 0.642  # vs small-preset baseline
print(f'big preset vs small preset: dExact = {delta_exact:+.3f}')
if r_greedy['exact_match_rate'] >= exact_target:
    print(f'WIN  Exact {r_greedy["exact_match_rate"]:.3f} >= {exact_target}: capacity was the bottleneck. Ship Stage 1; resume Stage 2 (ellipses).')
elif r_greedy['exact_match_rate'] >= 0.70:
    print(f'PARTIAL  Exact {r_greedy["exact_match_rate"]:.3f}: still improving. Try +50 epochs (warm-start from this checkpoint) or larger model.')
else:
    print(f'PLATEAU  Exact {r_greedy["exact_match_rate"]:.3f}: capacity is NOT the bottleneck. Diagnose: which orientations are the catastrophes? See docs/a3-gpu-handoff.md.')


## 5b. Beam search vs greedy on the *same* trained model

Greedy is myopic -- once it picks a bad token, the rest cascades (the row-3
catastrophe in the panel below). Beam-3 explores 3 paths and reduces that
rate. No retraining; just a different decoder. Slower than greedy on CPU but
comparable on GPU; expect ~+0.02 to ~+0.05 F1 and a bigger jump on
`exact_match_rate`.


In [ ]:
ar_beam = AutoregressivePredictor(model, C, max_tokens=cfg['max_seq_len'],
                                   device='cuda', beam_size=3)
r_beam = run_predictor(ar_beam, ds, C)
print('Predictor                       F1       Exact   Near')
print(f'AR greedy (this run)            {r_greedy["mean_f1"]:.3f}    {r_greedy["exact_match_rate"]:.3f}   {r_greedy["near_match_rate"]:.3f}')
print(f'AR beam-3 (this run)            {r_beam["mean_f1"]:.3f}    {r_beam["exact_match_rate"]:.3f}   {r_beam["near_match_rate"]:.3f}')
print()
delta_f1 = r_beam['mean_f1'] - r_greedy['mean_f1']
delta_exact = r_beam['exact_match_rate'] - r_greedy['exact_match_rate']
print(f'beam vs greedy: dF1 = {delta_f1:+.3f}  dExact = {delta_exact:+.3f}')
# use the better predictor for the visualization cell
if r_beam['mean_f1'] > r_greedy['mean_f1']:
    ar = ar_beam
    print('-> using beam-3 for the visualization below')


## 6. Eyeball the wireframes

Six held-out boxes: input | model prediction | ground truth.


In [ ]:
import matplotlib.pyplot as plt
from lines.datagen.render import render_primitives

N = 6
fig, axes = plt.subplots(N, 3, figsize=(7, 2.2 * N))
for k in range(N):
    img, gt = ds[k]
    pred = ar(img)
    pred_img = render_primitives(pred, C.width, C.height)
    gt_img = render_primitives(gt, C.width, C.height)
    for ax, im, title in zip(axes[k], [img, pred_img, gt_img],
                              ['input', 'prediction', 'ground truth']):
        ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title if k == 0 else ''); ax.axis('off')
plt.tight_layout(); plt.show()


## 7. (Optional) Save the trained checkpoint to Google Drive

Colab sessions get reaped. Uncomment to persist the checkpoint.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/lines_checkpoints
# !cp checkpoints/ar_box64_gpu/model.pt /content/drive/MyDrive/lines_checkpoints/ar_box64.pt
# print('saved to Drive')
